<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/ROSG_Test6g_compliance_weighted_Zeff_calibration_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROSG Test6g — Compliance-weighted Zeff calibration audit

This notebook tests whether the i=5 crossover instability can be explained by compliance or conductance-rigidity broadening.

It compares a rigid dyadic baseline with compliance-weighted corrections before any final Test7 claim.

## Imports, configuration, output

In [ ]:
# ============================================================
# ROSG Test6g — Compliance-weighted Z_eff calibration audit
# ============================================================
# Purpose:
#   Test whether the i=5 Zeff/crossover instability detected by
#   Test6f V2/V3 can be explained by compliance of conductance channels.
#
# Main idea:
#   Rigid dyadic baseline:
#       Z_c(i) = Z0 + i ln(2)
#
#   Compliance-weighted baseline:
#       Z_c(i) = Z0 + i ln(2) + chi * C_mean(i)
#
#   Dispersive compliance baseline:
#       Z_c(i) = Z0 + i ln(2) + chi * C_mean(i) + lambda * C_std(i)
#
# Methodological guardrail:
#   Compliance must be measured independently from Z_c whenever possible,
#   preferably from conductance/rigidity/compliance outputs. If only spectral
#   proxies exist, the result is exploratory and cannot be used as proof.
# ============================================================

import os, re, io, json, math, zipfile, shutil, hashlib, time, warnings
from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.signal import savgol_filter
    SCIPY_SIGNAL_AVAILABLE = True
except Exception:
    SCIPY_SIGNAL_AVAILABLE = False


@dataclass
class Test6gConfig:
    project_name: str = 'ROSG_Test6g_compliance_weighted_Zeff_calibration_audit'
    out_dir: str = '/content/ROSG_Test6g_compliance_weighted_Zeff_calibration_audit'
    use_google_drive: bool = True
    drive_out_dir: str = '/content/drive/MyDrive/ROSG_exports/ROSG_Test6g_compliance_weighted_Zeff_calibration_audit'

    search_roots: tuple = (
        '/content/drive/MyDrive/ROSG_exports',
        '/content/drive/MyDrive',
        '/content',
        '/mnt/data',
    )

    # Source patterns
    test6d_patterns: tuple = ('Test6d_V2_i3_i4_checkpoint_DSI_transmission_audit', 'Test6d')
    test6e_patterns: tuple = ('Test6e_i5_stochastic_heat_trace_DSI_transmission_audit', 'Test6e')
    test6f_patterns: tuple = ('Test6fV3', 'Test6f')
    compliance_patterns: tuple = ('compliance', 'rigidity', 'energy_rigidity', 'conductance', 'Conductance', 'Test2bis', 'rg_conductances')

    scenarios: tuple = ('H0_smooth', 'H1_lattice_coherent_A002')
    observables: tuple = ('ds_eff_median', 'ds_eff_mean', 'logP_mid', 'c_eff_mean')
    spectral_observables: tuple = ('ds_eff_median', 'ds_eff_mean', 'logP_mid')

    # Preferred crossover definition from Test6fV3 insight
    preferred_z_method: str = 'Z_slope_peak'  # slope_peak best captures H1 dyadic front
    alternative_z_method: str = 'Z_half_range'

    min_unique_Z: int = 9
    publication_min_Z_for_i5: int = 60
    boundary_fraction_reject: float = 0.06
    smoothing_window: int = 7
    smoothing_poly: int = 2

    alpha0: float = math.log(2)
    alpha_area: float = 2*math.log(2)

    # Numeric safeguards
    ridge: float = 1e-9
    min_points_for_claim: int = 4  # with i=3,4,5 only, verdict must stay provisional

CFG = Test6gConfig()


def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def prepare_output_dir(cfg=CFG):
    out = Path(cfg.out_dir)
    if cfg.use_google_drive and in_colab():
        try:
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive', force_remount=False)
            if Path('/content/drive/MyDrive').exists():
                out = Path(cfg.drive_out_dir)
        except Exception as exc:
            print('[Drive warning]', exc)
    return ensure_output_subdirs(out)


def ensure_output_subdirs(out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    for sub in ['tables','figures','metadata','extracts']:
        (out_dir/sub).mkdir(parents=True, exist_ok=True)
    return out_dir

OUT_DIR = prepare_output_dir(CFG)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Guardrails and source discovery

In [ ]:
# ============================================================
# Guardrails and source discovery
# ============================================================

def run_autotests(cfg=CFG):
    print('Running Test6g autotests...')
    assert abs(cfg.alpha0 - math.log(2)) < 1e-15
    assert abs(cfg.alpha_area - 2*math.log(2)) < 1e-15
    assert cfg.min_unique_Z >= 5
    print('All autotests passed.')


def guardrail_report(cfg=CFG):
    return {
        'metric_regime': 'hbar*G*c^-3 = l_P^2 is an area in metric units.',
        'pre_metric_ROSG': 'Use Planck quantities as calibration ratios, not raw powers of area.',
        'test6g_question': 'Does independently measured compliance explain the residual shift of Z_c(i) away from i ln2?',
        'anti_overfit_rule': 'Compliance-weighted improvement is not a proof if compliance is inferred from the same Z_c curve being fitted.',
        'small_sample_warning': 'With only i=3,4,5, any compliance fit is provisional; add i=6 or independent compliance outputs for claims.',
    }


def classify_zip(path, cfg=CFG):
    s = str(path)
    tags=[]
    if any(p in s for p in cfg.test6d_patterns): tags.append('test6d')
    if any(p in s for p in cfg.test6e_patterns): tags.append('test6e')
    if any(p in s for p in cfg.test6f_patterns): tags.append('test6f')
    if any(p in s for p in cfg.compliance_patterns): tags.append('compliance_candidate')
    return '+'.join(sorted(set(tags)))


def discover_zip_files(cfg=CFG):
    rows=[]; seen=set()
    for root_s in cfg.search_roots:
        root=Path(root_s)
        if not root.exists(): continue
        try:
            for p in root.rglob('*.zip'):
                src=classify_zip(p,cfg)
                if not src: continue
                key=str(p.resolve()) if p.exists() else str(p)
                if key in seen: continue
                seen.add(key)
                rows.append({'path':str(p),'name':p.name,'size_bytes':int(p.stat().st_size),'source_type':src,'mtime':float(p.stat().st_mtime)})
        except Exception:
            continue
    df=pd.DataFrame(rows)
    if len(df):
        # prefer exact Test6d/e plus compliance candidates; newest first
        df=df.sort_values(['source_type','mtime','size_bytes'],ascending=[True,False,False]).reset_index(drop=True)
    return df


def read_csvs_from_zip(zip_path, source_type):
    rows=[]; tables={}
    try:
        with zipfile.ZipFile(zip_path,'r') as zf:
            for m in zf.namelist():
                if not m.lower().endswith('.csv'): continue
                try:
                    with zf.open(m) as f:
                        df=pd.read_csv(f)
                    key=f'{Path(zip_path).name}::{m}'
                    tables[key]=df
                    rows.append({'archive':str(zip_path),'source_type':source_type,'member':m,'key':key,'n_rows':len(df),'n_cols':len(df.columns),'columns':'|'.join(map(str,df.columns))})
                except Exception as exc:
                    rows.append({'archive':str(zip_path),'source_type':source_type,'member':m,'key':'','n_rows':0,'n_cols':0,'columns':f'READ_ERROR: {exc}'})
    except Exception as exc:
        rows.append({'archive':str(zip_path),'source_type':source_type,'member':'','key':'','n_rows':0,'n_cols':0,'columns':f'ZIP_ERROR: {exc}'})
    return pd.DataFrame(rows), tables


def extract_all_csvs(zip_catalog):
    cats=[]; tables={}; source_by_key={}
    for _,row in zip_catalog.iterrows():
        cat,tt=read_csvs_from_zip(row['path'],row['source_type'])
        cats.append(cat)
        for k,df in tt.items():
            tables[k]=df
            source_by_key[k]=row['source_type']
    return (pd.concat(cats,ignore_index=True) if cats else pd.DataFrame()), tables, source_by_key

## Crossover extraction

In [ ]:
# ============================================================
# Crossover extraction, based on Test6fV3 but compact
# ============================================================

def clean_numeric_curve(df,z_col,y_col):
    tmp=df[[z_col,y_col]].copy(); tmp.columns=['Z','y']
    tmp['Z']=pd.to_numeric(tmp['Z'],errors='coerce')
    tmp['y']=pd.to_numeric(tmp['y'],errors='coerce')
    tmp=tmp.replace([np.inf,-np.inf],np.nan).dropna()
    if len(tmp)==0: return tmp
    return tmp.groupby('Z',as_index=False).agg(y=('y','mean')).sort_values('Z').reset_index(drop=True)


def smooth_y(y,cfg=CFG):
    y=np.asarray(y,dtype=float); n=len(y)
    if n<5: return y
    if SCIPY_SIGNAL_AVAILABLE:
        w=min(cfg.smoothing_window, n if n%2 else n-1)
        if w>=5:
            try: return savgol_filter(y,window_length=w,polyorder=min(cfg.smoothing_poly,w-2),mode='interp')
            except Exception: pass
    k=min(5,n); return np.convolve(y,np.ones(k)/k,mode='same')


def curve_metrics(curve,cfg=CFG):
    if len(curve)<cfg.min_unique_Z: return None
    Z=curve['Z'].to_numpy(float); y=curve['y'].to_numpy(float)
    if np.any(np.diff(Z)<=0): return None
    ys=smooth_y(y,cfg)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        dy=np.gradient(ys,Z)
    finite=np.isfinite(Z)&np.isfinite(ys)&np.isfinite(dy)
    if finite.sum()<cfg.min_unique_Z: return None
    n=len(Z); lo=max(1,int(np.floor(cfg.boundary_fraction_reject*n))); hi=min(n-2,int(np.ceil((1-cfg.boundary_fraction_reject)*n)))
    valid=np.arange(n); valid=valid[(valid>=lo)&(valid<=hi)&finite]
    if len(valid)==0: return None
    idx=valid[np.argmax(np.abs(dy[valid]))]
    ymin,ymax=float(np.nanmin(ys)),float(np.nanmax(ys)); yr=ymax-ymin
    if not np.isfinite(yr) or yr<=1e-12: return None
    idx_half=int(np.nanargmin(np.abs(ys-(ymin+0.5*yr))))
    return {'Z_slope_peak':float(Z[idx]),'Z_half_range':float(Z[idx_half]),'slope_peak_abs':float(abs(dy[idx])),'y_range':float(yr),'n_unique_Z':int(n),'Z_min':float(Z.min()),'Z_max':float(Z.max())}


def is_iteration_table(df,cfg=CFG):
    cols=set(map(str,df.columns))
    return {'i','scenario','Z'}.issubset(cols) and any(o in cols for o in cfg.observables)


def extract_iteration_candidates(tables,source_by_key,cfg=CFG):
    rows=[]
    for key,df in tables.items():
        src=source_by_key.get(key,'')
        if not any(t in src for t in ['test6d','test6e']): continue
        if not is_iteration_table(df,cfg): continue
        for scenario in cfg.scenarios:
            sdf=df[df['scenario'].astype(str)==scenario].copy()
            if len(sdf)==0: continue
            for i_val,gi in sdf.groupby('i'):
                try: i_int=int(i_val)
                except Exception: continue
                for rank,obs in enumerate(cfg.observables):
                    if obs not in gi.columns: continue
                    curve=clean_numeric_curve(gi,'Z',obs)
                    met=curve_metrics(curve,cfg)
                    if met is None: continue
                    rows.append({'source_key':key,'source_type':src,'i':i_int,'scenario':scenario,'observable':obs,'observable_rank':rank,'is_spectral':obs in cfg.spectral_observables,**met})
    out=pd.DataFrame(rows)
    if len(out):
        out=out.sort_values(['scenario','i','n_unique_Z','observable_rank','slope_peak_abs'],ascending=[True,True,False,True,False]).reset_index(drop=True)
    return out


def select_best_candidates(candidates,cfg=CFG):
    rows=[]
    if candidates.empty: return candidates
    for (scenario,i,obs),g in candidates.groupby(['scenario','i','observable']):
        gg=g.copy()
        if int(i)==5:
            pub=gg[gg['n_unique_Z']>=cfg.publication_min_Z_for_i5]
            if len(pub): gg=pub
        rows.append(gg.sort_values(['n_unique_Z','slope_peak_abs','y_range'],ascending=[False,False,False]).iloc[0].to_dict())
    return pd.DataFrame(rows).sort_values(['scenario','i','observable_rank']).reset_index(drop=True)


def build_zc_consensus(best,cfg=CFG):
    rows=[]
    for scenario in cfg.scenarios:
        for method in [cfg.preferred_z_method,cfg.alternative_z_method]:
            sub=best[(best['scenario']==scenario)&(best['is_spectral']==True)]
            for i,gi in sub.groupby('i'):
                rows.append({'scenario':scenario,'i':int(i),'method':method,'Z_c':float(np.median(gi[method])),'Z_mean':float(np.mean(gi[method])),'Z_std':float(np.std(gi[method])),'n_obs':int(len(gi)),'observables':'|'.join(map(str,gi['observable'].tolist()))})
    out=pd.DataFrame(rows)
    if len(out): out=out.sort_values(['scenario','method','i']).reset_index(drop=True)
    return out

## Compliance extraction

In [ ]:
# ============================================================
# Compliance extraction
# ============================================================

COMPLIANCE_PATTERNS = ('compliance','C_eff','Cmean','C_std','flexibility','softness')
RIGIDITY_PATTERNS = ('rigidity','stiffness','K_eff','Kmean','energy_rigidity')
CONDUCTANCE_PATTERNS = ('conductance','cond','g_v','g_eff','weight','c_eff')


def numeric_cols(df):
    return [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]


def infer_i_column(df):
    for c in ['i','level_i','iteration','iter','n','level']:
        if c in df.columns: return c
    return None


def infer_scenario_column(df):
    for c in ['scenario','case','label','condition']:
        if c in df.columns: return c
    return None


def classify_feature_col(col):
    s=str(col).lower()
    if any(p.lower() in s for p in COMPLIANCE_PATTERNS): return 'explicit_compliance'
    if any(p.lower() in s for p in RIGIDITY_PATTERNS): return 'rigidity_inverse_proxy'
    if any(p.lower() in s for p in CONDUCTANCE_PATTERNS): return 'conductance_inverse_proxy'
    return None


def extract_compliance_features(tables,source_by_key,cfg=CFG):
    rows=[]
    raw=[]
    for key,df in tables.items():
        src=source_by_key.get(key,'')
        # Search all tables, but prioritize compliance candidates and Test6 conductance-like outputs.
        i_col=infer_i_column(df)
        if i_col is None: continue
        scen_col=infer_scenario_column(df)
        ncols=numeric_cols(df)
        feature_cols=[]
        for c in ncols:
            kind=classify_feature_col(c)
            if kind:
                feature_cols.append((c,kind))
        if not feature_cols: continue
        tmp=df.copy()
        tmp['_i']=pd.to_numeric(tmp[i_col],errors='coerce')
        tmp=tmp.dropna(subset=['_i'])
        tmp['_i']=tmp['_i'].astype(int)
        if scen_col is not None:
            tmp['_scenario']=tmp[scen_col].astype(str)
        else:
            tmp['_scenario']='ALL'
        for (scenario,i),g in tmp.groupby(['_scenario','_i']):
            for col,kind in feature_cols:
                vals=pd.to_numeric(g[col],errors='coerce').replace([np.inf,-np.inf],np.nan).dropna()
                if len(vals)==0: continue
                vals=vals.astype(float)
                mean=float(vals.mean()); std=float(vals.std(ddof=0)); med=float(vals.median())
                if kind=='explicit_compliance':
                    c_mean=mean; c_std=std; direction='direct'
                elif kind in ('rigidity_inverse_proxy','conductance_inverse_proxy'):
                    # Higher conductance/rigidity means less compliance; use inverse positive scale.
                    pos=vals[vals>0]
                    if len(pos)==0: continue
                    inv=1.0/(pos + cfg.ridge)
                    c_mean=float(inv.mean()); c_std=float(inv.std(ddof=0)); direction='inverse'
                else:
                    continue
                rows.append({'source_key':key,'source_type':src,'scenario':scenario,'i':int(i),'feature_col':col,'feature_kind':kind,'direction':direction,'C_mean_raw':c_mean,'C_std_raw':c_std,'raw_mean':mean,'raw_std':std,'raw_median':med,'n':int(len(vals))})
    feat=pd.DataFrame(rows)
    if feat.empty: return feat, pd.DataFrame()
    # Normalize each feature_col across i to z-score, then aggregate by i/scenario.
    feat['C_mean_norm']=np.nan
    feat['C_std_norm']=np.nan
    for col,idx in feat.groupby('feature_col').groups.items():
        sub=feat.loc[idx]
        for src_col,dst_col in [('C_mean_raw','C_mean_norm'),('C_std_raw','C_std_norm')]:
            x=sub[src_col].astype(float)
            mu=x.mean(); sd=x.std(ddof=0)
            if not np.isfinite(sd) or sd<1e-12:
                feat.loc[idx,dst_col]=0.0
            else:
                feat.loc[idx,dst_col]=(x-mu)/sd
    # Scenario matching: keep exact scenario plus ALL. Aggregate by i and scenario-class.
    agg_rows=[]
    for scenario in list(cfg.scenarios)+['ALL']:
        use=feat[(feat['scenario']==scenario) | (feat['scenario']=='ALL')].copy()
        if len(use)==0: continue
        for i,g in use.groupby('i'):
            agg_rows.append({'scenario':scenario,'i':int(i),'C_mean':float(g['C_mean_norm'].mean()),'C_std':float(g['C_std_norm'].mean()),'C_feature_count':int(len(g)),'C_source_keys':'|'.join(sorted(set(map(str,g['source_key'])))),'C_feature_kinds':'|'.join(sorted(set(map(str,g['feature_kind']))))})
    agg=pd.DataFrame(agg_rows)
    if len(agg): agg=agg.sort_values(['scenario','i']).reset_index(drop=True)
    return feat, agg

## Model fitting

In [ ]:
# ============================================================
# Model fitting
# ============================================================

def standardize_column(x):
    x=np.asarray(x,dtype=float)
    mu=np.nanmean(x); sd=np.nanstd(x)
    if not np.isfinite(sd) or sd<1e-12: return np.zeros_like(x), mu, sd
    return (x-mu)/sd, mu, sd


def fit_linear_model(df, model_name, cfg=CFG):
    # Models:
    # M0: fixed ln2, intercept only for residual z - i ln2
    # M1: free alpha, intercept + i
    # M2: fixed ln2 + C_mean
    # M3: fixed ln2 + C_mean + C_std
    y=df['Z_c'].to_numpy(float)
    i=df['i'].to_numpy(float)
    base=cfg.alpha0*i
    n=len(y)
    if model_name=='M0_dyadic_ln2':
        X=np.ones((n,1)); target=y-base; params=['Z0']
    elif model_name=='M1_alpha_free':
        X=np.column_stack([np.ones(n), i]); target=y; params=['Z0','alpha']
    elif model_name=='M2_ln2_plus_Cmean':
        X=np.column_stack([np.ones(n), df['C_mean'].to_numpy(float)]); target=y-base; params=['Z0','chi_Cmean']
    elif model_name=='M3_ln2_plus_Cmean_Cstd':
        X=np.column_stack([np.ones(n), df['C_mean'].to_numpy(float), df['C_std'].to_numpy(float)]); target=y-base; params=['Z0','chi_Cmean','lambda_Cstd']
    else:
        raise ValueError(model_name)
    try:
        beta,*_=np.linalg.lstsq(X,target,rcond=None)
        pred_target=X@beta
        pred=pred_target+base if model_name!='M1_alpha_free' else pred_target
        resid=y-pred
        rss=float(np.sum(resid**2)); rmse=float(np.sqrt(rss/max(n,1)))
        k=X.shape[1]
        # AIC/AICc safeguards for tiny n.
        aic=float(n*np.log(max(rss/n,1e-12))+2*k) if n>0 else np.nan
        if n-k-1>0:
            aicc=float(aic + (2*k*(k+1))/(n-k-1))
        else:
            aicc=np.nan
        return {'model':model_name,'n':int(n),'k':int(k),'params':json.dumps({p:float(b) for p,b in zip(params,beta)},ensure_ascii=False),'rss':rss,'rmse':rmse,'aic':aic,'aicc':aicc,'pred':pred,'resid':resid}
    except Exception as exc:
        return {'model':model_name,'n':int(n),'k':None,'params':json.dumps({'error':str(exc)}),'rss':np.nan,'rmse':np.nan,'aic':np.nan,'aicc':np.nan,'pred':np.full(n,np.nan),'resid':np.full(n,np.nan)}


def loocv_rmse(df, model_name, cfg=CFG):
    if len(df)<3: return np.nan
    errs=[]
    for idx in df.index:
        train=df.drop(index=idx)
        test=df.loc[[idx]]
        f=fit_linear_model(train,model_name,cfg)
        params=json.loads(f['params'])
        if 'error' in params: continue
        i=float(test['i'].iloc[0]); z=float(test['Z_c'].iloc[0]); base=cfg.alpha0*i
        if model_name=='M0_dyadic_ln2': pred=params['Z0']+base
        elif model_name=='M1_alpha_free': pred=params['Z0']+params['alpha']*i
        elif model_name=='M2_ln2_plus_Cmean': pred=params['Z0']+base+params['chi_Cmean']*float(test['C_mean'].iloc[0])
        elif model_name=='M3_ln2_plus_Cmean_Cstd': pred=params['Z0']+base+params['chi_Cmean']*float(test['C_mean'].iloc[0])+params['lambda_Cstd']*float(test['C_std'].iloc[0])
        else: continue
        errs.append((z-pred)**2)
    if not errs: return np.nan
    return float(np.sqrt(np.mean(errs)))


def compare_compliance_models(zc, comp_agg, cfg=CFG):
    rows=[]; pred_rows=[]
    if zc.empty or comp_agg.empty: return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    for scenario in cfg.scenarios:
        for method in [cfg.preferred_z_method, cfg.alternative_z_method]:
            zz=zc[(zc['scenario']==scenario)&(zc['method']==method)].copy()
            cc=comp_agg[(comp_agg['scenario']==scenario) | (comp_agg['scenario']=='ALL')].copy()
            # Prefer exact scenario if available; otherwise ALL.
            if len(comp_agg[comp_agg['scenario']==scenario]): cc=comp_agg[comp_agg['scenario']==scenario].copy()
            else: cc=comp_agg[comp_agg['scenario']=='ALL'].copy()
            df=zz.merge(cc[['i','C_mean','C_std','C_feature_count','C_feature_kinds']],on='i',how='inner')
            df=df.dropna(subset=['Z_c','C_mean','C_std'])
            label='H0' if scenario=='H0_smooth' else 'H1'
            if len(df)<2:
                rows.append({'label':label,'scenario':scenario,'method':method,'verdict':'insufficient_compliance_matched_points','n':int(len(df))})
                continue
            for model in ['M0_dyadic_ln2','M1_alpha_free','M2_ln2_plus_Cmean','M3_ln2_plus_Cmean_Cstd']:
                f=fit_linear_model(df,model,cfg)
                loocv=loocv_rmse(df,model,cfg)
                row={k:v for k,v in f.items() if k not in ['pred','resid']}
                row.update({'label':label,'scenario':scenario,'method':method,'loocv_rmse':loocv,'n_matched':int(len(df)),'C_feature_kinds':'|'.join(sorted(set(map(str,df['C_feature_kinds']))))})
                rows.append(row)
                for idx,orig in enumerate(df.to_dict(orient='records')):
                    pred_rows.append({'label':label,'scenario':scenario,'method':method,'model':model,'i':orig['i'],'Z_c':orig['Z_c'],'C_mean':orig['C_mean'],'C_std':orig['C_std'],'pred':float(f['pred'][idx]) if np.isfinite(f['pred'][idx]) else np.nan,'resid':float(f['resid'][idx]) if np.isfinite(f['resid'][idx]) else np.nan})
    comp=pd.DataFrame(rows)
    preds=pd.DataFrame(pred_rows)
    verdicts=[]
    if len(comp):
        for (scenario,method),g in comp.groupby(['scenario','method']):
            gg=g[pd.to_numeric(g.get('rss',np.nan),errors='coerce').notna()].copy()
            label='H0' if scenario=='H0_smooth' else 'H1'
            if len(gg)==0:
                verdicts.append({'label':label,'scenario':scenario,'method':method,'verdict':'insufficient_fit'})
                continue
            m0=gg[gg['model']=='M0_dyadic_ln2']
            m2=gg[gg['model']=='M2_ln2_plus_Cmean']
            m3=gg[gg['model']=='M3_ln2_plus_Cmean_Cstd']
            best=gg.sort_values('rss').iloc[0]
            m0_rss=float(m0['rss'].iloc[0]) if len(m0) else np.nan
            best_comp=gg[gg['model'].isin(['M2_ln2_plus_Cmean','M3_ln2_plus_Cmean_Cstd'])].sort_values('rss').head(1)
            if len(best_comp) and np.isfinite(m0_rss) and m0_rss>0:
                improvement=1-float(best_comp['rss'].iloc[0])/m0_rss
                comp_model=str(best_comp['model'].iloc[0])
            else:
                improvement=np.nan; comp_model='none'
            if len(gg)<4:
                status='provisional_small_n'
            elif np.isfinite(improvement) and improvement>0.35:
                status='compliance_improves_dyadic_fit'
            else:
                status='no_clear_compliance_gain'
            verdicts.append({'label':label,'scenario':scenario,'method':method,'verdict':status,'best_model':str(best['model']),'best_compliance_model':comp_model,'rss_improvement_vs_M0':float(improvement) if np.isfinite(improvement) else np.nan,'n':int(gg['n_matched'].max()) if 'n_matched' in gg else None})
    return comp, preds, pd.DataFrame(verdicts)

## Figures and package export

In [ ]:
# ============================================================
# Figures and package export
# ============================================================

def make_figures(zc,comp_agg,model_preds,model_verdicts):
    fig_dir=OUT_DIR/'figures'
    if len(zc):
        for scenario in CFG.scenarios:
            label='H0' if scenario=='H0_smooth' else 'H1'
            sub=zc[zc['scenario']==scenario]
            if len(sub):
                plt.figure(figsize=(8,5))
                for method in [CFG.preferred_z_method,CFG.alternative_z_method]:
                    pts=sub[sub['method']==method].sort_values('i')
                    if len(pts): plt.plot(pts['i'],pts['Z_c'],marker='o',label=method)
                plt.title(f'Test6g — {label} crossover definitions')
                plt.xlabel('IFS level i'); plt.ylabel('Z_c')
                plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
                plt.savefig(fig_dir/f'test6g_{label}_zc_methods.png',dpi=180); plt.close()
    if len(comp_agg):
        for scenario in list(CFG.scenarios)+['ALL']:
            sub=comp_agg[comp_agg['scenario']==scenario].sort_values('i')
            if len(sub):
                label='ALL' if scenario=='ALL' else ('H0' if scenario=='H0_smooth' else 'H1')
                plt.figure(figsize=(7,4.5))
                plt.plot(sub['i'],sub['C_mean'],marker='o',label='C_mean norm')
                plt.plot(sub['i'],sub['C_std'],marker='o',label='C_std norm')
                plt.title(f'Test6g — {label} compliance features')
                plt.xlabel('IFS level i'); plt.ylabel('normalized compliance proxy')
                plt.grid(alpha=.3); plt.legend(); plt.tight_layout()
                plt.savefig(fig_dir/f'test6g_{label}_compliance_features.png',dpi=180); plt.close()
    if len(model_preds):
        for (scenario,method),g in model_preds.groupby(['scenario','method']):
            label='H0' if scenario=='H0_smooth' else 'H1'
            plt.figure(figsize=(8,5))
            actual=g.drop_duplicates('i').sort_values('i')
            plt.scatter(actual['i'],actual['Z_c'],s=70,label='observed Z_c')
            for model,gm in g.groupby('model'):
                if model in ['M0_dyadic_ln2','M2_ln2_plus_Cmean','M3_ln2_plus_Cmean_Cstd','M1_alpha_free']:
                    gm=gm.sort_values('i')
                    plt.plot(gm['i'],gm['pred'],marker='x',label=model)
            plt.title(f'Test6g — {label} {method} compliance fit')
            plt.xlabel('IFS level i'); plt.ylabel('Z_c')
            plt.grid(alpha=.3); plt.legend(fontsize=8); plt.tight_layout()
            plt.savefig(fig_dir/f'test6g_{label}_{method}_model_fit.png',dpi=180); plt.close()


def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''): h.update(chunk)
    return h.hexdigest()


def write_readme_manifest_zip(report):
    ensure_output_subdirs(OUT_DIR)
    readme=f"""# ROSG Test6g — Compliance-weighted Zeff calibration audit

## Purpose

Test whether the i=5 crossover instability can be explained by compliance of conductance channels.

## Compared models

- M0: `Z_c(i)=Z0+i ln2`
- M1: `Z_c(i)=Z0+i alpha`
- M2: `Z_c(i)=Z0+i ln2+chi C_mean(i)`
- M3: `Z_c(i)=Z0+i ln2+chi C_mean(i)+lambda C_std(i)`

## Global verdict

```text
{report.get('global_verdict','not_run')}
```

## Test7 recommendation

```text
{report.get('test7_recommendation','')}
```

## Methodological boundary

Compliance must be independently measured from conductance/rigidity/compliance outputs where possible. With only i=3,4,5, results are provisional.
"""
    (OUT_DIR/'README.md').write_text(readme,encoding='utf-8')
    rows=[]
    for p in sorted(OUT_DIR.rglob('*')):
        if p.is_file() and p.name!='manifest_sha256.csv':
            rows.append({'path':str(p.relative_to(OUT_DIR)),'size_bytes':int(p.stat().st_size),'sha256':sha256_file(p)})
    pd.DataFrame(rows).to_csv(OUT_DIR/'manifest_sha256.csv',index=False)
    return shutil.make_archive(str(OUT_DIR),'zip',OUT_DIR)

## Main audit

In [ ]:
# ============================================================
# Main audit
# ============================================================

def run_full_audit(cfg=CFG):
    ensure_output_subdirs(OUT_DIR)
    run_autotests(cfg)
    (OUT_DIR/'metadata'/'guardrail.json').write_text(json.dumps(guardrail_report(cfg),indent=2,ensure_ascii=False),encoding='utf-8')

    zips=discover_zip_files(cfg)
    zips.to_csv(OUT_DIR/'tables'/'source_zip_catalog.csv',index=False)
    print(f'[discovery] ZIP sources found: {len(zips)}')

    csv_catalog,tables,source_by_key=extract_all_csvs(zips)
    csv_catalog.to_csv(OUT_DIR/'tables'/'csv_catalog.csv',index=False)
    print(f'[extraction] CSV tables found: {len(tables)}')

    candidates=extract_iteration_candidates(tables,source_by_key,cfg)
    candidates.to_csv(OUT_DIR/'tables'/'crossover_candidates.csv',index=False)
    best=select_best_candidates(candidates,cfg)
    best.to_csv(OUT_DIR/'tables'/'best_crossover_candidates.csv',index=False)
    zc=build_zc_consensus(best,cfg)
    zc.to_csv(OUT_DIR/'tables'/'zc_consensus.csv',index=False)
    print(f'[crossover] consensus points: {len(zc)}')

    feat_raw, comp_agg=extract_compliance_features(tables,source_by_key,cfg)
    feat_raw.to_csv(OUT_DIR/'tables'/'compliance_raw_features.csv',index=False)
    comp_agg.to_csv(OUT_DIR/'tables'/'compliance_features_by_i.csv',index=False)
    print(f'[compliance] raw features: {len(feat_raw)} aggregated rows: {len(comp_agg)}')

    model_comp, model_preds, model_verdicts=compare_compliance_models(zc,comp_agg,cfg)
    model_comp.to_csv(OUT_DIR/'tables'/'compliance_weighted_model_comparison.csv',index=False)
    model_preds.to_csv(OUT_DIR/'tables'/'compliance_weighted_model_predictions.csv',index=False)
    model_verdicts.to_csv(OUT_DIR/'tables'/'compliance_weighted_verdicts.csv',index=False)

    make_figures(zc,comp_agg,model_preds,model_verdicts)

    # Global verdict logic
    has_compliance=len(comp_agg)>0
    exact_or_conductance=False
    if len(feat_raw):
        kinds='|'.join(sorted(set(feat_raw['feature_kind'].astype(str))))
        exact_or_conductance=('explicit_compliance' in kinds) or ('conductance_inverse_proxy' in kinds) or ('rigidity_inverse_proxy' in kinds)
    h1_pref=model_verdicts[(model_verdicts.get('label','')=='H1') & (model_verdicts.get('method','')==cfg.preferred_z_method)] if len(model_verdicts) else pd.DataFrame()
    improved=False
    if len(h1_pref):
        imp=pd.to_numeric(h1_pref['rss_improvement_vs_M0'],errors='coerce').max()
        improved=np.isfinite(imp) and imp>0.25
    if not has_compliance:
        global_verdict='insufficient_independent_compliance_data'
        test7_recommendation='Do not launch final Test7. Generate/load compliance or conductance-rigidity outputs first.'
    elif improved and exact_or_conductance:
        global_verdict='conditional_compliance_broadening_supported'
        test7_recommendation='Draft Test7-provisional with compliance-broadened ln2 prior; still add i=6 or independent cross-calibration before claims.'
    elif has_compliance and exact_or_conductance:
        global_verdict='compliance_data_found_but_no_decisive_broadening_gain'
        test7_recommendation='Do not claim compliance correction yet; inspect features and add i=6 or stronger compliance outputs.'
    else:
        global_verdict='only_weak_or_circular_compliance_proxy_available'
        test7_recommendation='Do not use compliance to justify Test7 unless independent conductance/rigidity outputs are supplied.'

    report={
        'status':'completed',
        'global_verdict':global_verdict,
        'test7_recommendation':test7_recommendation,
        'source_zip_count':int(len(zips)),
        'csv_table_count':int(len(tables)),
        'crossover_candidate_count':int(len(candidates)),
        'zc_consensus_count':int(len(zc)),
        'compliance_raw_feature_count':int(len(feat_raw)),
        'compliance_aggregated_count':int(len(comp_agg)),
        'model_verdicts':model_verdicts.to_dict(orient='records') if len(model_verdicts) else [],
        'guardrail':guardrail_report(cfg),
        'created_time':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime()),
    }
    (OUT_DIR/'metadata'/'test6g_report.json').write_text(json.dumps(report,indent=2,ensure_ascii=False),encoding='utf-8')
    zip_path=write_readme_manifest_zip(report)
    return {'zip_catalog':zips,'csv_catalog':csv_catalog,'candidates':candidates,'best':best,'zc':zc,'feat_raw':feat_raw,'comp_agg':comp_agg,'model_comp':model_comp,'model_preds':model_preds,'model_verdicts':model_verdicts,'report':report,'zip_path':zip_path}

## Run Test6g

In [ ]:
results = run_full_audit(CFG)

print(json.dumps({
    'status': results['report']['status'],
    'global_verdict': results['report']['global_verdict'],
    'test7_recommendation': results['report']['test7_recommendation'],
    'zip_path': results['zip_path'],
    'compliance_raw_feature_count': results['report']['compliance_raw_feature_count'],
    'compliance_aggregated_count': results['report']['compliance_aggregated_count'],
}, indent=2, ensure_ascii=False))

display(results['zip_catalog'])
display(results['zc'])
display(results['feat_raw'].head(50))
display(results['comp_agg'])
display(results['model_comp'])
display(results['model_verdicts'])
display(pd.DataFrame([results['report']]))


Running Test6g autotests...
All autotests passed.
[discovery] ZIP sources found: 11
[extraction] CSV tables found: 159
[crossover] consensus points: 12
[compliance] raw features: 140 aggregated rows: 15
{
  "status": "completed",
  "global_verdict": "conditional_compliance_broadening_supported",
  "test7_recommendation": "Draft Test7-provisional with compliance-broadened ln2 prior; still add i=6 or independent cross-calibration before claims.",
  "zip_path": "/content/drive/MyDrive/ROSG_exports/ROSG_Test6g_compliance_weighted_Zeff_calibration_audit.zip",
  "compliance_raw_feature_count": 140,
  "compliance_aggregated_count": 15
}


,path,name,size_bytes,source_type,mtime
0,/content/drive/MyDrive/ROSG_exports/ROSG_Test5...,ROSG_Test5_V3b_IFS_conductance_point_reconstru...,439482,compliance_candidate,1.780600e+09
1,/content/drive/MyDrive/ROSG_exports/ROSG_Test5...,ROSG_Test5_V3_IFS_conductance_point_reconstruc...,340387,compliance_candidate,1.780594e+09
2,/content/drive/MyDrive/ROSG_exports/ROSG_Test5...,ROSG_Test5_V2_IFS_conductance_point_reconstruc...,249491,compliance_candidate,1.780593e+09
3,/content/drive/MyDrive/ROSG_energy_rigidity_it...,ROSG_energy_rigidity_iteration_calibration_out...,602232,compliance_candidate,1.779369e+09
4,/content/drive/MyDrive/ROSG_energy_rigidity/RO...,ROSG_energy_rigidity_outputs.zip,223073,compliance_candidate,1.779178e+09
5,/content/drive/MyDrive/ROSG_exports/piste2bis_...,piste2bis_rg_conductances_cell4_N50_outputs.zip,984972,compliance_candidate,1.779089e+09
6,/content/drive/MyDrive/ROSG_exports/ROSG_Test6...,ROSG_Test6d_V2_i3_i4_checkpoint_DSI_transmissi...,3564203,test6d,1.781368e+09
7,/content/drive/MyDrive/ROSG_exports/ROSG_Test6...,ROSG_Test6e_i5_stochastic_heat_trace_DSI_trans...,1790108,test6e,1.780673e+09
8,/content/drive/MyDrive/ROSG_exports/ROSG_Test6...,ROSG_Test6fV3_crossover_definition_and_i5_reca...,241962,test6f,1.782031e+09
9,/content/drive/MyDrive/ROSG_exports/ROSG_Test6...,ROSG_Test6fV2_clean_Zeff_iteration_calibration...,450661,test6f,1.781993e+09


,scenario,i,method,Z_c,Z_mean,Z_std,n_obs,observables
0,H0_smooth,3,Z_half_range,1.06250,1.09375,0.192638,3,ds_eff_median|ds_eff_mean|logP_mid
1,H0_smooth,4,Z_half_range,1.25000,1.18750,0.088388,3,ds_eff_median|ds_eff_mean|logP_mid
2,H0_smooth,5,Z_half_range,1.62500,1.50000,0.246063,3,ds_eff_median|ds_eff_mean|logP_mid
3,H0_smooth,3,Z_slope_peak,0.78125,0.84375,0.088388,3,ds_eff_median|ds_eff_mean|logP_mid
4,H0_smooth,4,Z_slope_peak,0.78125,0.78125,0.153093,3,ds_eff_median|ds_eff_mean|logP_mid
5,H0_smooth,5,Z_slope_peak,4.43750,3.31250,1.590990,3,ds_eff_median|ds_eff_mean|logP_mid
6,H1_lattice_coherent_A002,3,Z_half_range,1.06250,1.06250,0.153093,3,ds_eff_median|ds_eff_mean|logP_mid
7,H1_lattice_coherent_A002,4,Z_half_range,1.06250,1.00000,0.233854,3,ds_eff_median|ds_eff_mean|logP_mid
8,H1_lattice_coherent_A002,5,Z_half_range,1.34375,1.34375,0.153093,3,ds_eff_median|ds_eff_mean|logP_mid
9,H1_lattice_coherent_A002,3,Z_slope_peak,-0.15625,0.31250,0.662913,3,ds_eff_median|ds_eff_mean|logP_mid


,source_key,source_type,scenario,i,feature_col,feature_kind,direction,C_mean_raw,C_std_raw,raw_mean,raw_std,raw_median,n,C_mean_norm,C_std_norm
0,ROSG_Test5_V3b_IFS_conductance_point_reconstru...,compliance_candidate,ALL,1,c_eff_mean,explicit_compliance,direct,2.168355e+00,7.653970e-01,2.168355e+00,7.653970e-01,2.303872e+00,17,-2.890122,1.794836
1,ROSG_Test5_V3b_IFS_conductance_point_reconstru...,compliance_candidate,ALL,1,c_eff_std,explicit_compliance,direct,2.481675e-16,1.847168e-16,2.481675e-16,1.847168e-16,2.220446e-16,17,-2.446873,-2.522208
2,ROSG_Test5_V3b_IFS_conductance_point_reconstru...,compliance_candidate,ALL,1,lambda1_weighted,conductance_inverse_proxy,inverse,5.371037e-01,2.215378e-01,2.168355e+00,7.653970e-01,2.303872e+00,17,-0.710468,-0.710719
3,ROSG_Test5_V3b_IFS_conductance_point_reconstru...,compliance_candidate,ALL,2,c_eff_mean,explicit_compliance,direct,2.191305e+00,7.616707e-01,2.191305e+00,7.616707e-01,2.350328e+00,17,-1.084215,1.316168
4,ROSG_Test5_V3b_IFS_conductance_point_reconstru...,compliance_candidate,ALL,2,c_eff_std,explicit_compliance,direct,1.873846e-02,1.337538e-02,1.873846e-02,1.337538e-02,1.634176e-02,17,-1.306208,-1.190411
5,ROSG_Test5_V3b_IFS_conductance_point_reconstru...,compliance_candidate,ALL,2,lambda1_weighted,conductance_inverse_proxy,inverse,1.387423e+00,5.708098e-01,8.369906e-01,2.909366e-01,8.977157e-01,17,-0.671204,-0.671315
6,ROSG_Test5_V3b_IFS_conductance_point_reconstru...,compliance_candidate,ALL,3,c_eff_mean,explicit_compliance,direct,2.202315e+00,7.591281e-01,2.202315e+00,7.591281e-01,2.371427e+00,17,-0.217778,0.989557
7,ROSG_Test5_V3b_IFS_conductance_point_reconstru...,compliance_candidate,ALL,3,c_eff_std,explicit_compliance,direct,3.342213e-02,2.390099e-02,3.342213e-02,2.390099e-02,2.751703e-02,17,-0.412369,-0.142367
8,ROSG_Test5_V3b_IFS_conductance_point_reconstru...,compliance_candidate,ALL,3,lambda1_weighted,conductance_inverse_proxy,inverse,4.364315e+00,1.791250e+00,2.656255e-01,9.156434e-02,2.860161e-01,17,-0.533743,-0.533627
9,ROSG_Test5_V3b_IFS_conductance_point_reconstru...,compliance_candidate,ALL,4,c_eff_mean,explicit_compliance,direct,2.206523e+00,7.576943e-01,2.206523e+00,7.576943e-01,2.378732e+00,17,0.113293,0.805370


,scenario,i,C_mean,C_std,C_feature_count,C_source_keys,C_feature_kinds
0,ALL,1,-2.015821,-0.479364,12,ROSG_Test5_V3_IFS_conductance_point_reconstruc...,conductance_inverse_proxy|explicit_compliance
1,ALL,2,-1.020542,-0.181852,12,ROSG_Test5_V3_IFS_conductance_point_reconstruc...,conductance_inverse_proxy|explicit_compliance
2,ALL,3,-0.313628,-0.014490,20,ROSG_Test5_V2_IFS_conductance_point_reconstruc...,conductance_inverse_proxy|explicit_compliance
3,ALL,4,-0.049186,0.331938,20,ROSG_Test5_V2_IFS_conductance_point_reconstruc...,conductance_inverse_proxy|explicit_compliance
4,ALL,5,0.725058,0.743969,20,ROSG_Test5_V2_IFS_conductance_point_reconstruc...,conductance_inverse_proxy|explicit_compliance
5,H0_smooth,1,-2.015821,-0.479364,12,ROSG_Test5_V3_IFS_conductance_point_reconstruc...,conductance_inverse_proxy|explicit_compliance
6,H0_smooth,2,-1.020542,-0.181852,12,ROSG_Test5_V3_IFS_conductance_point_reconstruc...,conductance_inverse_proxy|explicit_compliance
7,H0_smooth,3,-0.268137,-0.094349,24,ROSG_Test5_V2_IFS_conductance_point_reconstruc...,conductance_inverse_proxy|explicit_compliance
8,H0_smooth,4,0.033367,0.237133,24,ROSG_Test5_V2_IFS_conductance_point_reconstruc...,conductance_inverse_proxy|explicit_compliance
9,H0_smooth,5,0.705602,0.578818,26,ROSG_Test5_V2_IFS_conductance_point_reconstruc...,conductance_inverse_proxy|explicit_compliance


,model,n,k,params,rss,rmse,aic,aicc,label,scenario,method,loocv_rmse,n_matched,C_feature_kinds
0,M0_dyadic_ln2,3,1,"{""Z0"": -0.7725887222397811}",4.804377e+00,1.265488e+00,3.412745,7.412745,H0,H0_smooth,Z_slope_peak,1.898231,3,conductance_inverse_proxy|explicit_compliance
1,M1_alpha_free,3,2,"{""Z0"": -5.312499999999998, ""alpha"": 1.82812499...",2.228027e+00,8.617864e-01,3.107513,NaN,H0,H0_smooth,Z_slope_peak,3.166405,3,conductance_inverse_proxy|explicit_compliance
2,M2_ln2_plus_Cmean,3,2,"{""Z0"": -1.1929306132987294, ""chi_Cmean"": 2.678...",1.239336e+00,6.427379e-01,1.347890,NaN,H0,H0_smooth,Z_slope_peak,2.964504,3,conductance_inverse_proxy|explicit_compliance
3,M3_ln2_plus_Cmean_Cstd,3,3,"{""Z0"": 0.35952543121594255, ""chi_Cmean"": 10.17...",6.754622e-30,1.500513e-15,-76.893063,NaN,H0,H0_smooth,Z_slope_peak,2.792841,3,conductance_inverse_proxy|explicit_compliance
4,M0_dyadic_ln2,3,1,"{""Z0"": -1.4600887222397814}",3.451779e-01,3.392039e-01,-4.486922,-0.486922,H0,H0_smooth,Z_half_range,0.508806,3,conductance_inverse_proxy|explicit_compliance
5,M1_alpha_free,3,2,"{""Z0"": 0.18749999999999922, ""alpha"": 0.28125}",5.859375e-03,4.419417e-02,-14.714974,NaN,H0,H0_smooth,Z_half_range,0.162380,3,conductance_inverse_proxy|explicit_compliance
6,M2_ln2_plus_Cmean,3,2,"{""Z0"": -1.3370907113350463, ""chi_Cmean"": -0.78...",3.992820e-02,1.153664e-01,-8.957854,NaN,H0,H0_smooth,Z_half_range,0.532105,3,conductance_inverse_proxy|explicit_compliance
7,M3_ln2_plus_Cmean_Cstd,3,3,"{""Z0"": -1.0584371085515984, ""chi_Cmean"": 0.561...",2.958228e-31,3.140185e-16,-76.893063,NaN,H0,H0_smooth,Z_half_range,0.318597,3,conductance_inverse_proxy|explicit_compliance
8,M0_dyadic_ln2,3,1,"{""Z0"": -1.9913387222397814}",3.297890e-01,3.315564e-01,-4.623744,-0.623744,H1,H1_lattice_coherent_A002,Z_slope_peak,0.497335,3,conductance_inverse_proxy|explicit_compliance
9,M1_alpha_free,3,2,"{""Z0"": -2.03125, ""alpha"": 0.703125}",3.295898e-01,3.314563e-01,-2.625556,NaN,H1,H1_lattice_coherent_A002,Z_slope_peak,1.217848,3,conductance_inverse_proxy|explicit_compliance


,label,scenario,method,verdict,best_model,best_compliance_model,rss_improvement_vs_M0,n
0,H0,H0_smooth,Z_half_range,compliance_improves_dyadic_fit,M3_ln2_plus_Cmean_Cstd,M3_ln2_plus_Cmean_Cstd,1.0,3
1,H0,H0_smooth,Z_slope_peak,compliance_improves_dyadic_fit,M3_ln2_plus_Cmean_Cstd,M3_ln2_plus_Cmean_Cstd,1.0,3
2,H1,H1_lattice_coherent_A002,Z_half_range,compliance_improves_dyadic_fit,M3_ln2_plus_Cmean_Cstd,M3_ln2_plus_Cmean_Cstd,1.0,3
3,H1,H1_lattice_coherent_A002,Z_slope_peak,compliance_improves_dyadic_fit,M3_ln2_plus_Cmean_Cstd,M3_ln2_plus_Cmean_Cstd,1.0,3


,status,global_verdict,test7_recommendation,source_zip_count,csv_table_count,crossover_candidate_count,zc_consensus_count,compliance_raw_feature_count,compliance_aggregated_count,model_verdicts,guardrail,created_time
0,completed,conditional_compliance_broadening_supported,Draft Test7-provisional with compliance-broade...,11,159,56,12,140,15,"[{'label': 'H0', 'scenario': 'H0_smooth', 'met...",{'metric_regime': 'hbar*G*c^-3 = l_P^2 is an a...,2026-06-21T10:17:34Z
